In [3]:
import pandas as pd
import re


In [4]:
df=pd.read_csv('cleaneddata.csv')
df 

,Unnamed: 0,cleaned_tweet,label
0,0,woman shouldnt complain cleaning house amp man...,2
1,1,boy dat coldtyga dwn bad cuffin dat hoe place,1
2,2,dawg ever fuck bitch start cry confused shit,1
3,3,look like tranny,1
4,4,shit hear might true might faker bitch told,1
...,...,...,...
24778,24778,yous muthafin lie right trash mine bible scrip...,1
24779,24779,youve gone broke wrong heart baby drove rednec...,2
24780,24780,young buck wan eat dat nigguh like aint fuckin...,1
24781,24781,youu got wild bitch tellin lie,1


In [5]:
df.columns

Index(['Unnamed: 0', 'cleaned_tweet', 'label'], dtype='object')

In [6]:
df.drop(columns=['Unnamed: 0'],inplace=True)

In [7]:
df

,cleaned_tweet,label
0,woman shouldnt complain cleaning house amp man...,2
1,boy dat coldtyga dwn bad cuffin dat hoe place,1
2,dawg ever fuck bitch start cry confused shit,1
3,look like tranny,1
4,shit hear might true might faker bitch told,1
...,...,...
24778,yous muthafin lie right trash mine bible scrip...,1
24779,youve gone broke wrong heart baby drove rednec...,2
24780,young buck wan eat dat nigguh like aint fuckin...,1
24781,youu got wild bitch tellin lie,1


In [8]:
X=df['cleaned_tweet']
Y=df['label']


In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)
print(X_train.shape)
print(Y_train.shape)

print(X_test.shape)
print(Y_test.shape)

(19826,)
(19826,)
(4957,)
(4957,)


In [21]:
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [11]:
max_features = 10000  # Number of words to consider as features
maxlen = 500

In [14]:
X_train = X_train.fillna("").astype(str)
X_test  = X_test.fillna("").astype(str)

In [15]:
max_features = 10000   # increased vocab size

tokenizer = Tokenizer(num_words=max_features, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

In [16]:
max_len = 100

X_train_padded = pad_sequences(X_train_seq, maxlen=max_len)
X_test_padded  = pad_sequences(X_test_seq, maxlen=max_len)

In [22]:
# 🔹 Model (model2)
model2 = Sequential([
    Embedding(max_features, 64),   # removed deprecated input_length
    SimpleRNN(64),
    Dropout(0.5),                  # 🔥 prevents overfitting
    Dense(3, activation='softmax') # 3 classes
])

# 🔹 Compile
model2.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# 🔹 Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# 🔹 Train
history = model2.fit(
    X_train_padded,
    Y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_padded, Y_test),
    callbacks=[early_stop]
)


Epoch 1/5
620/620 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.8495 - loss: 0.4356 - val_accuracy: 0.8858 - val_loss: 0.3195
Epoch 2/5
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.9155 - loss: 0.2449 - val_accuracy: 0.8852 - val_loss: 0.3298
Epoch 3/5
620/620 ━━━━━━━━━━━━━━━━━━━━ 15s 25ms/step - accuracy: 0.9481 - loss: 0.1547 - val_accuracy: 0.8745 - val_loss: 0.3963


In [23]:
model2.save('model2.h5')

In [ ]:
import pickle



# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)